In [ ]:
# ==============================
# Google Drive Torrent Downloader (Resumable + Multi-magnet)
# ==============================

!apt-get install -y python3-libtorrent > /dev/null

import libtorrent as lt
import time, os, threading
from google.colab import drive
from tqdm import tqdm
from IPython.display import clear_output

# ==============================
# 1. Mount Google Drive
# ==============================
drive.mount('/content/drive')

# Change this path to where you want torrents saved in Drive
base_path = "/content/drive/MyDrive/TorrentDownloads"
os.makedirs(base_path, exist_ok=True)

# ==============================
# 2. Torrent Downloader Class
# ==============================
class TorrentDownloader:
    def __init__(self, magnet_link, save_path):
        self.magnet_link = magnet_link
        self.save_path = save_path
        self.session = lt.session()
        self.session.listen_on(6881, 6891)

        params = {
            'save_path': self.save_path,
            'storage_mode': lt.storage_mode_t(2),
        }
        self.handle = lt.add_magnet_uri(self.session, self.magnet_link, params)

    def start_download(self):
        while not self.handle.has_metadata():
            print("⏳ Fetching metadata...")
            time.sleep(2)

    def get_progress(self):
        return self.handle.status().progress

    def is_complete(self):
        return self.handle.status().is_seeding

    def stop_download(self):
        self.session.pause()

# ==============================
# 3. Resumable Download Function
# ==============================
def download_torrent_resumable(magnet_link):
    torrent_name = magnet_link.split("&dn=")[-1] if "&dn=" in magnet_link else magnet_link[:20]
    torrent_name = torrent_name.replace("+", "_").replace("%20", "_")
    download_folder = os.path.join(base_path, torrent_name)
    os.makedirs(download_folder, exist_ok=True)

    print(f"\n\033[92m[START] {torrent_name}\033[0m")

    torrent = TorrentDownloader(magnet_link, download_folder)

    if os.listdir(download_folder):
        print(f"\033[93m[RESUME] Found existing files. Resuming {torrent_name}...\033[0m")

    torrent.start_download()

    with tqdm(total=100, desc=torrent_name, unit='%') as pbar:
        last_progress = 0
        while not torrent.is_complete():
            try:
                current_progress = int(torrent.get_progress() * 100)
                if current_progress > last_progress:
                    pbar.update(current_progress - last_progress)
                    last_progress = current_progress
                time.sleep(5)
            except KeyboardInterrupt:
                print(f"\n⚡ Paused manually → Can resume later.")
                torrent.stop_download()
                return
            except Exception as e:
                print(f"\n\033[91m[ERROR] {torrent_name}: {e}\033[0m")
                break

    torrent.stop_download()
    if torrent.is_complete():
        print(f"\n\033[94m[DONE] {torrent_name}\033[0m")
    else:
        print(f"\n\033[91m[FAILED] {torrent_name}\033[0m")

# ==============================
# 4. Add Multiple Magnet Links
# ==============================
magnet_links = [
    # 🔽 Add your magnet links here
    "magnet:?xt=urn:btih:54ONS4YQUHWNDDTHEY4AX2CMIN5F6PYH&dn=Spiderman%202%20(2004)%201080p%20x264%20DD5.1%20EN%20NL%20Subs%20%5BAsian%20Planet%5D&tr=udp%3A%2F%2Ftracker.pomf.se%3A80%2Fannounce",
]

# ==============================
# 5. Start Downloads
# ==============================
for magnet in magnet_links:
    download_torrent_resumable(magnet)


Mounted at /content/drive

[START] Spiderman_2_(2004)_1080p_x264_DD5.1_EN_NL_Subs_%5BAsian_Planet%5D&tr=udp%3A%2F%2Ftracker.pomf.se%3A80%2Fannounce
⏳ Fetching metadata...


/tmp/ipython-input-998751931.py:30: DeprecationWarning: listen_on() is deprecated
  self.session.listen_on(6881, 6891)
/tmp/ipython-input-998751931.py:36: DeprecationWarning: add_magnet_uri() is deprecated
  self.handle = lt.add_magnet_uri(self.session, self.magnet_link, params)
/tmp/ipython-input-998751931.py:39: DeprecationWarning: has_metadata() is deprecated
  while not self.handle.has_metadata():
Spiderman_2_(2004)_1080p_x264_DD5.1_EN_NL_Subs_%5BAsian_Planet%5D&tr=udp%3A%2F%2Ftracker.pomf.se%3A80%2Fannounce:  99%|█████████▉| 99/100 [11:20<00:06,  6.87s/%]



[DONE] Spiderman_2_(2004)_1080p_x264_DD5.1_EN_NL_Subs_%5BAsian_Planet%5D&tr=udp%3A%2F%2Ftracker.pomf.se%3A80%2Fannounce
